# Ensemble of ensembles - model stacking

Two types of ensemble of ensembles:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import pandas as pd

In [2]:
df=pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")

In [3]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
num_col=list(df.describe().columns)
col_categorical=list(set(df.columns).difference(num_col))
remove_list=['EmployeeCount','EmployeeNumber','StandardHours']
col_numerical=[e for e in num_col if e not in remove_list]
attrition_to_num={'Yes':0,
                  'No':1}
df['Attrition_num']=df['Attrition'].map(attrition_to_num)
col_categorical.remove('Attrition')
df_cat = pd.get_dummies(df[col_categorical], dtype=int)
X=pd.concat([df[col_numerical],df_cat],axis=1)
y=df['Attrition_num']

In [5]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y)

In [6]:
from sklearn import preprocessing
from sklearn.model_selection import cross_val_score,cross_val_predict
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, roc_auc_score

In [7]:
def print_score(clf, X_train, X_test, y_train, y_test, train=True):
    
    lb = preprocessing.LabelBinarizer()
    lb.fit(y_train)
    
    if train:
        res = clf.predict(X_train)
        print("Train result:\n")
        print("Accuracy score: {0:.4f}\n".format(accuracy_score(y_train, res)))
        print("Classification report:\n{}\n".format(classification_report(y_train, res)))
        print("Confusion matrix:\n{}\n".format(confusion_matrix(y_train, res)))
        print("ROC AUC: {0:.4f}\n".format(
            roc_auc_score(lb.transform(y_train), lb.transform(res))
        ))
        res=cross_val_score(clf,X_train,y_train,cv=10,scoring='accuracy')
        print("Average accuracy :\t{0:.4f}".format(np.mean(res)))
        print("Accuracy SD:\t\t{0:.4f}".format(np.std(res)))
        
    else:
        res_test = clf.predict(X_test)
        print("Test result:\n")
        print("Accuracy score: {0:.4f}\n".format(accuracy_score(y_test, res_test)))
        print("Classification report:\n{}\n".format(classification_report(y_test, res_test)))
        print("Confusion matrix:\n{}\n".format(confusion_matrix(y_test, res_test)))
        print("ROC AUC: {0:.4f}\n".format(
            roc_auc_score(lb.transform(y_test), lb.transform(res_test))
        ))


## Model-1: Decision Tree

In [8]:
from sklearn.tree import DecisionTreeClassifier

In [9]:
tree_clf=DecisionTreeClassifier()
tree_clf.fit(X_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [10]:
print_score(tree_clf,X_train,X_test,y_train,y_test,train=True)
print("\n************************")
print_score(tree_clf,X_train,X_test,y_train,y_test,train=False)

Train result:

Accuracy score: 1.0000

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       182
           1       1.00      1.00      1.00       920

    accuracy                           1.00      1102
   macro avg       1.00      1.00      1.00      1102
weighted avg       1.00      1.00      1.00      1102


Confusion matrix:
[[182   0]
 [  0 920]]

ROC AUC: 1.0000

Average accuracy :	0.7632
Accuracy SD:		0.0174

************************
Test result:

Accuracy score: 0.7582

Classification report:
              precision    recall  f1-score   support

           0       0.27      0.36      0.31        55
           1       0.88      0.83      0.85       313

    accuracy                           0.76       368
   macro avg       0.58      0.60      0.58       368
weighted avg       0.79      0.76      0.77       368


Confusion matrix:
[[ 20  35]
 [ 54 259]]

ROC AUC: 0.5956



## Model-2: Random Forest

In [11]:
from sklearn.ensemble import RandomForestClassifier

In [12]:
rf_clf=RandomForestClassifier(n_estimators=100)
rf_clf.fit(X_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [13]:
print_score(rf_clf,X_train,X_test,y_train,y_test,train=True)
print("\n************************")
print_score(rf_clf,X_train,X_test,y_train,y_test,train=False)

Train result:

Accuracy score: 1.0000

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       182
           1       1.00      1.00      1.00       920

    accuracy                           1.00      1102
   macro avg       1.00      1.00      1.00      1102
weighted avg       1.00      1.00      1.00      1102


Confusion matrix:
[[182   0]
 [  0 920]]

ROC AUC: 1.0000

Average accuracy :	0.8448
Accuracy SD:		0.0158

************************
Test result:

Accuracy score: 0.8804

Classification report:
              precision    recall  f1-score   support

           0       0.79      0.27      0.41        55
           1       0.89      0.99      0.93       313

    accuracy                           0.88       368
   macro avg       0.84      0.63      0.67       368
weighted avg       0.87      0.88      0.85       368


Confusion matrix:
[[ 15  40]
 [  4 309]]

ROC AUC: 0.6300



In [14]:
en_en=pd.DataFrame()

In [15]:
tree_clf.predict_proba(X_train)

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [1., 0.],
       [1., 0.],
       [0., 1.]], shape=(1102, 2))

In [16]:
en_en['tree_clf']=pd.DataFrame(tree_clf.predict_proba(X_train))[1]
en_en['rf_clf']=pd.DataFrame(rf_clf.predict_proba(X_train))[1]
col_name=en_en.columns
en_en=pd.concat([en_en,pd.DataFrame(y_train).reset_index(drop=True)],axis=1)

In [17]:
en_en.head()

,tree_clf,rf_clf,Attrition_num
0,1.0,0.98,1
1,1.0,0.96,1
2,1.0,0.98,1
3,1.0,0.81,1
4,1.0,0.99,1


In [18]:
tmp=list(col_name)
tmp.append('ind')
en_en.columns=tmp

In [19]:
en_en.head()

,tree_clf,rf_clf,ind
0,1.0,0.98,1
1,1.0,0.96,1
2,1.0,0.98,1
3,1.0,0.81,1
4,1.0,0.99,1


## Meta Classifier

In [20]:
from sklearn.linear_model import LogisticRegression

In [23]:
m_clf=LogisticRegression(fit_intercept=False,solver='lbfgs')

In [24]:
m_clf.fit(en_en[['tree_clf','rf_clf']],en_en['ind'])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,False
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [25]:
en_test=pd.DataFrame()

In [27]:
en_test['tree_clf']=pd.DataFrame(tree_clf.predict_proba(X_train))[1]
en_test['rf_clf']=pd.DataFrame(rf_clf.predict_proba(X_train))[1]
col_name=en_en.columns
en_test['combined']=m_clf.predict(en_test[['tree_clf','rf_clf']])

In [28]:
col_name=en_test.columns
tmp=list(col_name)
tmp.append('ind')

In [29]:
tmp

['tree_clf', 'rf_clf', 'combined', 'ind']

In [30]:
en_test=pd.concat([en_test,pd.DataFrame(y_test).reset_index(drop=True)],axis=1)

In [31]:
en_test.columns=tmp

In [32]:
print(pd.crosstab(en_test['ind'],en_test['combined']))

combined   0    1
ind              
0.0       11   44
1.0       56  257


In [37]:
en_test = en_test.dropna(subset=['ind', 'combined'])

In [38]:
print(round(accuracy_score(en_test['ind'],en_test['combined']),4))

0.7283


In [39]:
 print(classification_report(en_test['ind'],en_test['combined']))

              precision    recall  f1-score   support

         0.0       0.16      0.20      0.18        55
         1.0       0.85      0.82      0.84       313

    accuracy                           0.73       368
   macro avg       0.51      0.51      0.51       368
weighted avg       0.75      0.73      0.74       368



# Single Classifier

In [40]:
df=pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")

In [41]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [42]:
df.Attrition.value_counts() /df.Attrition.count()

Attrition
No     0.838776
Yes    0.161224
Name: count, dtype: float64

In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier

In [45]:
class_weight={0:0.839,1:0.161}

In [46]:
pd.Series(list(y_train)).value_counts() /pd.Series(list(y_train)).count()

1    0.834846
0    0.165154
Name: count, dtype: float64

In [49]:
forest=RandomForestClassifier(class_weight=class_weight,n_estimators=100)

In [50]:
ada=AdaBoostClassifier(estimator=forest,n_estimators=100,
                       learning_rate=0.5,random_state=42)

In [52]:
ada.fit(X_train,y_train.ravel())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14440\3653582213.py:1: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  ada.fit(X_train,y_train.ravel())


,estimator,"RandomForestC...39, 1: 0.161})"
,n_estimators,100
,learning_rate,0.5
,algorithm,'deprecated'
,random_state,42
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [53]:
print_score(ada,X_train,X_test,y_train,y_test,train=True)
print("\n************************")
print_score(ada,X_train,X_test,y_train,y_test,train=False)

Train result:

Accuracy score: 1.0000

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       182
           1       1.00      1.00      1.00       920

    accuracy                           1.00      1102
   macro avg       1.00      1.00      1.00      1102
weighted avg       1.00      1.00      1.00      1102


Confusion matrix:
[[182   0]
 [  0 920]]

ROC AUC: 1.0000

Average accuracy :	0.8421
Accuracy SD:		0.0135

************************
Test result:

Accuracy score: 0.8696

Classification report:
              precision    recall  f1-score   support

           0       0.89      0.15      0.25        55
           1       0.87      1.00      0.93       313

    accuracy                           0.87       368
   macro avg       0.88      0.57      0.59       368
weighted avg       0.87      0.87      0.83       368


Confusion matrix:
[[  8  47]
 [  1 312]]

ROC AUC: 0.5711



In [54]:
bag_clf=BaggingClassifier(estimator=ada,n_estimators=50,
                          max_samples=1.0,max_features=1.0,bootstrap=True,
                          bootstrap_features=False,n_jobs=-1,
                          random_state=42)

In [55]:
bag_clf.fit(X_train,y_train.ravel())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14440\4228070472.py:1: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  bag_clf.fit(X_train,y_train.ravel())


,estimator,AdaBoostClass...ndom_state=42)
,n_estimators,50
,max_samples,1.0
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,-1
,random_state,42
,verbose,0


In [57]:
print_score(bag_clf,X_train,X_test,y_train,y_test,train=True)
print("\n************************")
print_score(bag_clf,X_train,X_test,y_train,y_test,train=False)

Train result:

Accuracy score: 0.9946

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98       182
           1       0.99      1.00      1.00       920

    accuracy                           0.99      1102
   macro avg       1.00      0.98      0.99      1102
weighted avg       0.99      0.99      0.99      1102


Confusion matrix:
[[176   6]
 [  0 920]]

ROC AUC: 0.9835

Average accuracy :	0.8430
Accuracy SD:		0.0088

************************
Test result:

Accuracy score: 0.8723

Classification report:
              precision    recall  f1-score   support

           0       0.90      0.16      0.28        55
           1       0.87      1.00      0.93       313

    accuracy                           0.87       368
   macro avg       0.89      0.58      0.60       368
weighted avg       0.88      0.87      0.83       368


Confusion matrix:
[[  9  46]
 [  1 312]]

ROC AUC: 0.5802

